In [1]:
# standard python tools
import os
import random
import copy
import time

# math and plotting
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit for the 10-fold CV
from sklearn.model_selection import StratifiedKFold

# pytorch setup
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU, BatchNorm1d, ModuleList, Parameter
from torch.utils.data import Subset

# pytorch geometric setup
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.typing import OptTensor, Optional
from torch_geometric.utils import degree, to_undirected, add_self_loops, softmax
from torch_geometric.nn.inits import glorot, zeros
from torch_geometric.nn import (
    GCNConv,
    GraphConv,
    GATv2Conv,
    GINConv,
    GINEConv,
    GATConv,
    TransformerConv,
    global_add_pool,
    global_mean_pool,
    MessagePassing
)

C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# run on cpu, using gpu for other experiments
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
# same as in experiment_overfitting_sum
class BaseGNN(nn.Module):
    def __init__(self, init_std=0.01):
        super().__init__()
        self.init_std = init_std

    def init_params(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_normal_(param, gain=self.init_std)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

class StandardGraphConv(BaseGNN):
    def __init__(self, in_channels, out_channels, num_layers, hidden_channels, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.convs = nn.ModuleList()
        self.convs.append(GraphConv(in_channels, hidden_channels, aggr='add', bias=bias))
        for _ in range(num_layers - 1):
            self.convs.append(GraphConv(hidden_channels, hidden_channels, aggr='add', bias=bias))

        self.pool = global_add_pool
        self.readout = nn.Linear(hidden_channels, out_channels, bias=bias)

        self.init_params_graphconv()

    def init_params_graphconv(self):
        for conv in self.convs:
            if hasattr(conv, 'lin_root'): nn.init.xavier_normal_(conv.lin_root.weight, gain=self.init_std)
            if hasattr(conv, 'lin_rel'): nn.init.constant_(conv.lin_rel.weight, 0.0)
        if hasattr(self.readout, 'weight'): nn.init.xavier_normal_(self.readout.weight, gain=self.init_std)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        x = self.pool(x, batch)
        return self.readout(x)

In [4]:
# sanity check
def print_feature_stats(dataset):
    xs = []
    for g in dataset:
        if g.x is not None:
            xs.append(g.x)
    xs = torch.cat(xs, dim=0)
    print("mean:", xs.mean(), "std:", xs.std(), "unique:", xs.unique())

In [5]:
def generate_dataset(name='Error', graph_type='Original'):
    original_dataset = TUDataset(root='data/TUDataset', name=name, use_node_attr=True)

    #print_feature_stats(original_dataset)   # CORRECT STATS

    if graph_type == 'Original':
        return original_dataset

    elif graph_type == 'Empty':
        empty_dataset = []

        for data in original_dataset:
            new_data = data.clone()

            # remove edges
            new_data.edge_index = torch.empty((2, 0), dtype=torch.long)

            # remove edge attributes if present
            if new_data.edge_attr is not None:
                new_data.edge_attr = torch.empty((0, new_data.edge_attr.size(1)),
                                                 dtype=new_data.edge_attr.dtype)

            empty_dataset.append(new_data)

        return empty_dataset

    else:
        raise ValueError(f"Unknown graph_type: {graph_type}")


In [6]:
import numpy as np

# standard EarlyStopper
class EarlyStopping:
    def __init__(self, patience=100, delta=0, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0  # Reset counter
        elif val_loss > self.best_loss + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print("Early stopping triggered.")

        return self.early_stop

In [7]:
# define a list of random seeds for 3 independent runs
SEEDS = [1, 2, 3]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

print(f"Experiment configured to run with {len(SEEDS)} seeds: {SEEDS}")

Experiment configured to run with 3 seeds: [1, 2, 3]


In [8]:
def train_model(model, train_loader, val_loader, epochs, patience):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    # early stopping seems to do strange things to the enzymes and proteins datasets.
    # early_stopper = EarlyStopping(patience=patience)

    best_val_loss = np.inf
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y.squeeze())
            loss.backward()
            optimizer.step()

        val_loss_sum = 0
        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                val_loss_sum += criterion(out, batch.y.squeeze()).item()

        avg_val_loss = val_loss_sum / len(val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()

        # if early_stopper(avg_val_loss):
            # break

    if best_model_state:
        model.load_state_dict(best_model_state)
    return model

In [9]:
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            pred = out.argmax(dim=1)
            correct += (pred == batch.y.squeeze()).sum().item()
            total += batch.y.size(0)
    return correct / total

In [10]:
def run_fold(dataset_list, train_idx, val_idx, test_idx, model_cls, hidden_channels, num_layers, epochs, patience):

    # create the subsets using the pre-split indices
    train_set = [dataset_list[i] for i in train_idx]
    val_set = [dataset_list[i] for i in val_idx]
    test_set = [dataset_list[i] for i in test_idx]

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

    IN_CHANNELS = dataset_list[0].x.size(1)
    OUT_CHANNELS = len(torch.unique(torch.tensor([data.y for data in dataset_list])))

    model = model_cls(IN_CHANNELS, OUT_CHANNELS, num_layers, hidden_channels).to(device)

    trained_model = train_model(model, train_loader, val_loader, epochs=epochs, patience=patience)

    test_acc = evaluate_model(trained_model, test_loader)

    return test_acc, trained_model

In [11]:
num_folds = 10
num_epochs = 1000
num_patience = 100
dataset_names = ['ENZYMES', 'PROTEINS']
graph_types = ['Empty', 'Original']
model_cls = StandardGraphConv
lr = 0.001
layers = 3
hidden_channels = 64
BATCH_SIZE = 8

results_all = []

print("--- Starting 10-Fold Stratified CV Experiment (3 Seeds) ---")
start_time = time.time()

# ---------------------------------------------------------
#    precompute all dataset variants once
#    (ENZYMES/PROTEINS × Original/Empty)
# ---------------------------------------------------------
datasets = {}
for name in dataset_names:
    for gtype in graph_types:
        print(f"Precomputing dataset: {name}, graph type: {gtype}")
        datasets[(name, gtype)] = generate_dataset(name=name, graph_type=gtype)

def build_folds(labels, num_folds, seed):
    indices = np.arange(len(labels))
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)
    rng = np.random.RandomState(seed)

    folds = []
    for train_val_idx, test_idx in skf.split(indices, labels):
        train_val_idx = train_val_idx.copy()
        rng.shuffle(train_val_idx)

        val_size = max(1, len(indices) // 10)
        val_idx = train_val_idx[:val_size]
        train_idx = train_val_idx[val_size:]

        folds.append((train_idx, val_idx, test_idx))

    return folds

all_folds = {}
for seed in SEEDS:
    for name in dataset_names:
        ds_orig = datasets[(name, 'Original')]
        labels = np.array([d.y.item() for d in ds_orig])
        all_folds[(name, seed)] = build_folds(labels, num_folds, seed)

for seed in SEEDS:
    print(f"\n==============================================")
    print(f"| STARTING NEW RUN: Seed = {seed:<3} |")
    print(f"==============================================")

    set_seed(seed)

    for name in dataset_names:
        print(f"\n===== Running Dataset: {name} =====")
        folds = all_folds[(name, seed)]

        for gtype in graph_types:
            print(f"--- Graph Type: {gtype} ---")
            ds = datasets[(name, gtype)]

            fold_accuracies = []

            for fold_idx, (train_idx, val_idx, test_idx) in enumerate(folds):
                test_acc, trained_model = run_fold(
                    ds,
                    train_idx,
                    val_idx,
                    test_idx,
                    model_cls,
                    hidden_channels=hidden_channels,
                    num_layers=layers,
                    epochs=num_epochs,
                    patience=num_patience
                )

                fold_accuracies.append(test_acc)

                layer = trained_model.convs[0]
                w1_norm = torch.norm(layer.lin_root.weight).item()
                w2_norm = torch.norm(layer.lin_rel.weight).item()
                ratio = w2_norm / w1_norm if w1_norm > 1e-6 else 0.0

                results_all.append({
                    'Seed': seed,
                    'Dataset': name,
                    'Graph Case': gtype,
                    'Fold': fold_idx + 1,
                    'Test Accuracy': test_acc,
                    'Weight Ratio (W2/W1)': ratio
                })

                print(f"  Fold {fold_idx+1}/{num_folds} | Acc: {test_acc:.4f} | Ratio: {ratio:.4f}")

            avg_acc = np.mean(fold_accuracies)
            std_acc = np.std(fold_accuracies)
            print(f"  -> AVG ACC (Seed {seed}): {avg_acc:.4f} ± {std_acc:.4f}")

end_time = time.time()
print(f"\nTotal Experiment Time: {end_time - start_time:.2f} seconds")
print(f"Total runs completed: {len(results_all)}")

df_results = pd.DataFrame(results_all)

final_summary = df_results.groupby(['Dataset', 'Graph Case'])['Test Accuracy'].agg(['mean', 'std'])

print("\n--- FINAL AGGREGATED RESULTS (Average over all seeds & folds) ---")
print(final_summary)

df_results.to_csv("rcov_proteins_enzymes_stratified_analysis.csv", index=False)

--- Starting 10-Fold Stratified CV Experiment (3 Seeds) ---
Precomputing dataset: ENZYMES, graph type: Empty
Precomputing dataset: ENZYMES, graph type: Original
Precomputing dataset: PROTEINS, graph type: Empty
Precomputing dataset: PROTEINS, graph type: Original

| STARTING NEW RUN: Seed = 1   |

===== Running Dataset: ENZYMES =====
--- Graph Type: Empty ---
  Fold 1/10 | Acc: 0.6000 | Ratio: 0.0000
  Fold 2/10 | Acc: 0.6833 | Ratio: 0.0000
  Fold 3/10 | Acc: 0.5000 | Ratio: 0.0000
  Fold 4/10 | Acc: 0.2667 | Ratio: 0.0000
  Fold 5/10 | Acc: 0.5000 | Ratio: 0.0000
  Fold 6/10 | Acc: 0.6167 | Ratio: 0.0000
  Fold 7/10 | Acc: 0.5833 | Ratio: 0.0000
  Fold 8/10 | Acc: 0.5667 | Ratio: 0.0000
  Fold 9/10 | Acc: 0.4167 | Ratio: 0.0000
  Fold 10/10 | Acc: 0.4167 | Ratio: 0.0000
  -> AVG ACC (Seed 1): 0.5150 ± 0.1161
--- Graph Type: Original ---
  Fold 1/10 | Acc: 0.1667 | Ratio: 1.1133
  Fold 2/10 | Acc: 0.2000 | Ratio: 1.0535
  Fold 3/10 | Acc: 0.2167 | Ratio: 1.0102
  Fold 4/10 | Acc: 0.33